### Phase 1: Load data 

The data is downloaded from Procurement KPI Analysis Dataset: Real-World Supplier Performance, Cost Savings, and Compliance Data.
kaggle : https://www.kaggle.com/datasets/shahriarkabir/procurement-kpi-analysis-dataset

### Step 1 — Update the file path

In [1]:
from pathlib import Path
import zipfile
import pandas as pd
import numpy as np

# Project folder based on your screenshot
PROJECT_DIR = Path.home() / "Desktop" / "Procurement KPI Analysis Dataset"

# Your actual Raw folder
RAW_DIR = PROJECT_DIR / "data" / "Raw"

# Original ZIP file
ZIP_PATH = RAW_DIR / "archive (2).zip"

# Confirm the path exists before proceeding
print("Project folder:", PROJECT_DIR)
print("ZIP file:", ZIP_PATH)
print("ZIP exists:", ZIP_PATH.exists())

if not ZIP_PATH.exists():
    raise FileNotFoundError(
        f"ZIP file not found at: {ZIP_PATH}"
    )

Project folder: C:\Users\User\Desktop\Procurement KPI Analysis Dataset
ZIP file: C:\Users\User\Desktop\Procurement KPI Analysis Dataset\data\Raw\archive (2).zip
ZIP exists: True


### Step 2: Load the raw CSV and inspect it

In [2]:
with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    csv_files = [
        name for name in archive.namelist()
        if name.lower().endswith(".csv")
    ]

    print("CSV files found:", csv_files)

    if len(csv_files) != 1:
        raise ValueError(
            f"Expected exactly one CSV file, found {len(csv_files)}."
        )

    with archive.open(csv_files[0]) as file:
        df_raw = pd.read_csv(file)

print("\nRaw dataset loaded successfully.")
print("Rows:", df_raw.shape[0])
print("Columns:", df_raw.shape[1])

display(df_raw.head())

CSV files found: ['Procurement KPI Analysis Dataset.csv']

Raw dataset loaded successfully.
Rows: 777
Columns: 11


,PO_ID,Supplier,Order_Date,Delivery_Date,Item_Category,Order_Status,Quantity,Unit_Price,Negotiated_Price,Defective_Units,Compliance
0,PO-00001,Alpha_Inc,2023-10-17,2023-10-25,Office Supplies,Cancelled,1176,20.13,17.81,NaN,Yes
1,PO-00002,Delta_Logistics,2022-04-25,2022-05-05,Office Supplies,Delivered,1509,39.32,37.34,235.0,Yes
2,PO-00003,Gamma_Co,2022-01-26,2022-02-15,MRO,Delivered,910,95.51,92.26,41.0,Yes
3,PO-00004,Beta_Supplies,2022-10-09,2022-10-28,Packaging,Delivered,1344,99.85,95.52,112.0,Yes
4,PO-00005,Delta_Logistics,2022-09-08,2022-09-20,Raw Materials,Delivered,1180,64.07,60.53,171.0,No


### Step 3 — Audit the complete dataset

In [4]:
# Reproducible raw-data quality audit
# 1. Dataset structure
print("DATASET SHAPE")
print(f"Rows: {df_raw.shape[0]}")
print(f"Columns: {df_raw.shape[1]}")

print("\nCOLUMN NAMES")
print(df_raw.columns.tolist())

print("\nDATA TYPES")
display(df_raw.dtypes.rename("Data_Type").to_frame())

# 2. Missing values
missing_audit = pd.DataFrame({
    "Missing_Count": df_raw.isna().sum(),
    "Missing_Percent": (df_raw.isna().mean() * 100).round(2)
})

print("\nMISSING VALUES BY COLUMN")
display(missing_audit)

# 3. Duplicate checks
print("\nEXACT DUPLICATE ROWS")
print(df_raw.duplicated().sum())

print("\nDUPLICATE PO_ID VALUES")
print(df_raw["PO_ID"].duplicated().sum())

# 4. Category checks
for column in [
    "Supplier",
    "Item_Category",
    "Order_Status",
    "Compliance"
]:
    print(f"\n{column} — VALUE COUNTS")
    display(df_raw[column].value_counts(dropna=False).to_frame("Count"))

# 5. Numeric summaries
numeric_columns = [
    "Quantity",
    "Unit_Price",
    "Negotiated_Price",
    "Defective_Units"
]

print("\nNUMERIC SUMMARY")
display(df_raw[numeric_columns].describe().T)

# 6. Date checks without changing the original columns
orders = df_raw.copy()

orders["Order_Date_Check"] = pd.to_datetime(
    orders["Order_Date"], errors="coerce"
)
orders["Delivery_Date_Check"] = pd.to_datetime(
    orders["Delivery_Date"], errors="coerce"
)

print("\nDATE PARSING CHECK")
print("Invalid/missing order dates:",
      orders["Order_Date_Check"].isna().sum())
print("Invalid/missing delivery dates:",
      orders["Delivery_Date_Check"].isna().sum())

print("\nDATE RANGE")
print("Earliest order:", orders["Order_Date_Check"].min())
print("Latest order:", orders["Order_Date_Check"].max())

# 7. Potential date anomalies
date_anomalies = orders[
    orders["Order_Date_Check"].notna()
    & orders["Delivery_Date_Check"].notna()
    & (
        orders["Delivery_Date_Check"]
        < orders["Order_Date_Check"]
    )
]

print("\nDELIVERY BEFORE ORDER DATE")
display(
    date_anomalies[
        ["PO_ID", "Order_Date", "Delivery_Date", "Order_Status"]
    ]
)

# 8. Defect counts exceeding order quantities
defect_anomalies = orders[
    orders["Defective_Units"].notna()
    & (
        (orders["Defective_Units"] < 0)
        | (orders["Defective_Units"] > orders["Quantity"])
    )
]

print("\nPOTENTIAL DEFECT-COUNT ANOMALIES")
display(
    defect_anomalies[
        ["PO_ID", "Supplier", "Quantity", "Defective_Units"]
    ]
)


DATASET SHAPE
Rows: 777
Columns: 11

COLUMN NAMES
['PO_ID', 'Supplier', 'Order_Date', 'Delivery_Date', 'Item_Category', 'Order_Status', 'Quantity', 'Unit_Price', 'Negotiated_Price', 'Defective_Units', 'Compliance']

DATA TYPES


,Data_Type
PO_ID,object
Supplier,object
Order_Date,object
Delivery_Date,object
Item_Category,object
Order_Status,object
Quantity,int64
Unit_Price,float64
Negotiated_Price,float64
Defective_Units,float64



MISSING VALUES BY COLUMN


,Missing_Count,Missing_Percent
PO_ID,0,0.0
Supplier,0,0.0
Order_Date,0,0.0
Delivery_Date,87,11.2
Item_Category,0,0.0
Order_Status,0,0.0
Quantity,0,0.0
Unit_Price,0,0.0
Negotiated_Price,0,0.0
Defective_Units,136,17.5



EXACT DUPLICATE ROWS
0

DUPLICATE PO_ID VALUES
0

Supplier — VALUE COUNTS


,Count
Supplier,
Delta_Logistics,171
Epsilon_Group,166
Beta_Supplies,156
Gamma_Co,143
Alpha_Inc,141


... (8 outputs left out)


DATE PARSING CHECK
Invalid/missing order dates: 0
Invalid/missing delivery dates: 87

DATE RANGE
Earliest order: 2022-01-01 00:00:00
Latest order: 2024-01-01 00:00:00

DELIVERY BEFORE ORDER DATE


,PO_ID,Order_Date,Delivery_Date,Order_Status
100,PO-00101,2022-02-27,2022-02-22,Delivered



POTENTIAL DEFECT-COUNT ANOMALIES


,PO_ID,Supplier,Quantity,Defective_Units


### Step 4 — Investigate missing values by order status
This is our first focused investigation. A missing defect count on a cancelled order may have a different explanation from a missing count on a delivered order.

In [5]:
# Missing-value patterns by order status

missing_by_status = (
    df_raw.groupby("Order_Status", dropna=False)
    .agg(
        Total_Orders=("PO_ID", "size"),
        Missing_Delivery_Dates=("Delivery_Date", lambda x: x.isna().sum()),
        Missing_Defect_Counts=("Defective_Units", lambda x: x.isna().sum())
    )
)

missing_by_status["Missing_Delivery_Pct"] = (
    missing_by_status["Missing_Delivery_Dates"]
    / missing_by_status["Total_Orders"] * 100
).round(2)

missing_by_status["Missing_Defect_Pct"] = (
    missing_by_status["Missing_Defect_Counts"]
    / missing_by_status["Total_Orders"] * 100
).round(2)

display(missing_by_status)


,Total_Orders,Missing_Delivery_Dates,Missing_Defect_Counts,Missing_Delivery_Pct,Missing_Defect_Pct
Order_Status,,,,,
Cancelled,63,8,9,12.70,14.29
Delivered,560,68,102,12.14,18.21
Partially Delivered,73,5,13,6.85,17.81
Pending,81,6,12,7.41,14.81


### Step 5 — Inspect the actual records with missing values
Let's look at individual purchase orders before making any cleaning decisions.

In [6]:
# Inspect records with missing delivery dates

missing_delivery = df_raw[
    df_raw["Delivery_Date"].isna()
].copy()

display(
    missing_delivery[
        [
            "PO_ID",
            "Supplier",
            "Order_Date",
            "Delivery_Date",
            "Order_Status",
            "Quantity",
            "Defective_Units",
            "Compliance"
        ]
    ].sort_values(["Order_Status", "PO_ID"])
)

print("Total missing delivery dates:", len(missing_delivery))


,PO_ID,Supplier,Order_Date,Delivery_Date,Order_Status,Quantity,Defective_Units,Compliance
68,PO-00069,Epsilon_Group,2022-08-27,NaN,Cancelled,63,NaN,Yes
94,PO-00095,Beta_Supplies,2023-10-18,NaN,Cancelled,1895,170.0,Yes
156,PO-00157,Epsilon_Group,2022-12-15,NaN,Cancelled,1913,61.0,Yes
218,PO-00219,Epsilon_Group,2023-04-08,NaN,Cancelled,606,16.0,Yes
270,PO-00271,Beta_Supplies,2022-12-14,NaN,Cancelled,1880,173.0,Yes
...,...,...,...,...,...,...,...,...
145,PO-00146,Gamma_Co,2023-04-25,NaN,Pending,468,24.0,Yes
379,PO-00380,Alpha_Inc,2022-09-08,NaN,Pending,636,18.0,Yes
446,PO-00447,Gamma_Co,2023-01-09,NaN,Pending,865,47.0,Yes
704,PO-00705,Delta_Logistics,2022-05-11,NaN,Pending,1326,NaN,Yes


Total missing delivery dates: 87


In [7]:
# Inspect records with missing delivery dates

missing_delivery = df_raw[
    df_raw["Delivery_Date"].isna()
].copy()

display(
    missing_delivery[
        [
            "PO_ID",
            "Supplier",
            "Order_Date",
            "Delivery_Date",
            "Order_Status",
            "Quantity",
            "Defective_Units",
            "Compliance"
        ]
    ].sort_values(["Order_Status", "PO_ID"])
)

print("Total missing delivery dates:", len(missing_delivery))


,PO_ID,Supplier,Order_Date,Delivery_Date,Order_Status,Quantity,Defective_Units,Compliance
68,PO-00069,Epsilon_Group,2022-08-27,NaN,Cancelled,63,NaN,Yes
94,PO-00095,Beta_Supplies,2023-10-18,NaN,Cancelled,1895,170.0,Yes
156,PO-00157,Epsilon_Group,2022-12-15,NaN,Cancelled,1913,61.0,Yes
218,PO-00219,Epsilon_Group,2023-04-08,NaN,Cancelled,606,16.0,Yes
270,PO-00271,Beta_Supplies,2022-12-14,NaN,Cancelled,1880,173.0,Yes
...,...,...,...,...,...,...,...,...
145,PO-00146,Gamma_Co,2023-04-25,NaN,Pending,468,24.0,Yes
379,PO-00380,Alpha_Inc,2022-09-08,NaN,Pending,636,18.0,Yes
446,PO-00447,Gamma_Co,2023-01-09,NaN,Pending,865,47.0,Yes
704,PO-00705,Delta_Logistics,2022-05-11,NaN,Pending,1326,NaN,Yes


Total missing delivery dates: 87


### Step 7 — Verify both missing-value filters
If an order is missing its delivery date, we need to investigate whether that is expected given its status. If its defect count is also missing, we need to investigate that separately. The two issues may overlap, but they are not necessarily the same problem.

In [8]:
# Verify missing-value filters independently

print("Missing Delivery_Date:", df_raw["Delivery_Date"].isna().sum())
print("Missing Defective_Units:", df_raw["Defective_Units"].isna().sum())

print("\nFirst 10 records with missing delivery dates:")
display(
    df_raw.loc[
        df_raw["Delivery_Date"].isna(),
        [
            "PO_ID", "Supplier", "Order_Date", "Delivery_Date",
            "Order_Status", "Quantity", "Defective_Units", "Compliance"
        ]
    ].head(10)
)

print("\nFirst 10 records with missing defect counts:")
display(
    df_raw.loc[
        df_raw["Defective_Units"].isna(),
        [
            "PO_ID", "Supplier", "Order_Date", "Delivery_Date",
            "Order_Status", "Quantity", "Defective_Units", "Compliance"
        ]
    ].head(10)
)


Missing Delivery_Date: 87
Missing Defective_Units: 136

First 10 records with missing delivery dates:


,PO_ID,Supplier,Order_Date,Delivery_Date,Order_Status,Quantity,Defective_Units,Compliance
13,PO-00014,Beta_Supplies,2022-02-02,NaN,Delivered,5000,NaN,Yes
29,PO-00030,Epsilon_Group,2023-08-27,NaN,Delivered,1005,30.0,Yes
39,PO-00040,Epsilon_Group,2022-12-11,NaN,Delivered,524,14.0,Yes
45,PO-00046,Alpha_Inc,2022-12-19,NaN,Delivered,1025,15.0,Yes
49,PO-00050,Alpha_Inc,2023-04-16,NaN,Pending,736,18.0,Yes
51,PO-00052,Delta_Logistics,2022-05-08,NaN,Delivered,612,102.0,Yes
56,PO-00057,Delta_Logistics,2023-10-06,NaN,Delivered,1578,206.0,Yes
68,PO-00069,Epsilon_Group,2022-08-27,NaN,Cancelled,63,NaN,Yes
75,PO-00076,Alpha_Inc,2022-06-16,NaN,Partially Delivered,141,NaN,Yes
78,PO-00079,Alpha_Inc,2022-08-03,NaN,Delivered,1528,31.0,No



First 10 records with missing defect counts:


,PO_ID,Supplier,Order_Date,Delivery_Date,Order_Status,Quantity,Defective_Units,Compliance
0,PO-00001,Alpha_Inc,2023-10-17,2023-10-25,Cancelled,1176,NaN,Yes
13,PO-00014,Beta_Supplies,2022-02-02,NaN,Delivered,5000,NaN,Yes
15,PO-00016,Beta_Supplies,2022-04-06,2022-04-21,Delivered,5000,NaN,Yes
17,PO-00018,Epsilon_Group,2022-08-27,2022-09-04,Delivered,921,NaN,Yes
19,PO-00020,Delta_Logistics,2023-09-09,2023-09-24,Delivered,180,NaN,No
21,PO-00022,Delta_Logistics,2023-07-29,2023-08-11,Delivered,1382,NaN,Yes
25,PO-00026,Epsilon_Group,2023-07-13,2023-07-24,Delivered,1487,NaN,Yes
30,PO-00031,Gamma_Co,2022-10-12,2022-10-26,Delivered,326,NaN,No
50,PO-00051,Epsilon_Group,2023-07-04,2023-07-14,Delivered,1007,NaN,Yes
57,PO-00058,Delta_Logistics,2023-09-26,2023-10-01,Delivered,1204,NaN,Yes


In [9]:
# Compare missing-value groups

missing_delivery_ids = set(
    df_raw.loc[df_raw["Delivery_Date"].isna(), "PO_ID"]
)

missing_defect_ids = set(
    df_raw.loc[df_raw["Defective_Units"].isna(), "PO_ID"]
)

print("Orders missing delivery dates:", len(missing_delivery_ids))
print("Orders missing defect counts:", len(missing_defect_ids))
print("Missing both:", len(missing_delivery_ids & missing_defect_ids))
print("Missing delivery date only:",
      len(missing_delivery_ids - missing_defect_ids))
print("Missing defect count only:",
      len(missing_defect_ids - missing_delivery_ids))


Orders missing delivery dates: 87
Orders missing defect counts: 136
Missing both: 15
Missing delivery date only: 72
Missing defect count only: 121


### Step 8 — What we've established
### Missing delivery dates
87 / 777
11.2% of orders
### Missing defect counts
136 / 777
17.5% of orders
### Missing both fields
15
Records needing joint review
### Missing at least one field
208
26.8% of all orders
The overlap is consistent: \(87 + 136 - 15 = 208\) distinct orders have at least one of these two fields missing.
Three observations matter:
- Some orders marked Delivered have no delivery date. We cannot calculate their delivery timeliness from the dates currently available.
- Some orders marked Cancelled have defect counts missing, which may be expected if no inspection occurred.
- Some delivered orders have defect counts missing even when a delivery date exists. Those require a separate explanation; we must not assume their defect count was zero.

### Step 9 — Determine which missing values are most suspicious
Let's now cross-tabulate the missing fields by order status. This will help us distinguish potentially expected missingness from records that need further investigation.

In [10]:

# Missing-value investigation by order status

audit = df_raw.copy()

audit["Missing_Delivery_Date"] = audit["Delivery_Date"].isna()
audit["Missing_Defect_Count"] = audit["Defective_Units"].isna()

audit["Missingness_Pattern"] = np.select(
    [
        audit["Missing_Delivery_Date"] & audit["Missing_Defect_Count"],
        audit["Missing_Delivery_Date"] & ~audit["Missing_Defect_Count"],
        ~audit["Missing_Delivery_Date"] & audit["Missing_Defect_Count"]
    ],
    [
        "Both missing",
        "Delivery date only",
        "Defect count only"
    ],
    default="Neither missing"
)

status_missingness = pd.crosstab(
    audit["Order_Status"],
    audit["Missingness_Pattern"],
    margins=True
)

display(status_missingness)

print("\nMissingness percentages within each order status:")

status_percentages = (
    pd.crosstab(
        audit["Order_Status"],
        audit["Missingness_Pattern"],
        normalize="index"
    ) * 100
).round(2)

display(status_percentages)


Missingness_Pattern,Both missing,Defect count only,Delivery date only,Neither missing,All
Order_Status,,,,,
Cancelled,1,8,7,47,63
Delivered,12,90,56,402,560
Partially Delivered,1,12,4,56,73
Pending,1,11,5,64,81
All,15,121,72,569,777



Missingness percentages within each order status:


Missingness_Pattern,Both missing,Defect count only,Delivery date only,Neither missing
Order_Status,,,,
Cancelled,1.59,12.70,11.11,74.60
Delivered,2.14,16.07,10.00,71.79
Partially Delivered,1.37,16.44,5.48,76.71
Pending,1.23,13.58,6.17,79.01


### Step 10 — Save an audit record
Once the results look correct, save the missingness report so the project has a reproducible record of the original data quality.

In [11]:
# Save audit outputs without changing the raw data

AUDIT_DIR = PROJECT_DIR / "outputs" / "audit"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

missing_audit.to_csv(
    AUDIT_DIR / "raw_missing_values_audit.csv",
    index=True
)

status_missingness.to_csv(
    AUDIT_DIR / "missingness_by_status_counts.csv"
)

status_percentages.to_csv(
    AUDIT_DIR / "missingness_by_status_percentages.csv"
)

print("Audit reports saved to:", AUDIT_DIR)


Audit reports saved to: C:\Users\User\Desktop\Procurement KPI Analysis Dataset\outputs\audit


### Step 11 — Audit numeric fields and business-rule violations
Now we'll inspect Quantity, Unit_Price, Negotiated_Price, and Defective_Units. We want to identify impossible values, unusual values, and assumptions that need confirmation before cleaning.

In [12]:
# Numeric data-quality checks
# This cell only reports issues; it does not modify df_raw.

numeric_cols = [
    "Quantity",
    "Unit_Price",
    "Negotiated_Price",
    "Defective_Units"
]

# Convert temporary copies to numeric for validation
numeric_check = df_raw[numeric_cols].apply(
    pd.to_numeric, errors="coerce"
)

print("1. Numeric conversion failures")
for col in numeric_cols:
    failures = df_raw[col].notna() & numeric_check[col].isna()
    print(f"{col}: {failures.sum()}")

print("\n2. Negative values")
display((numeric_check < 0).sum().rename("Negative_Count").to_frame())

print("\n3. Zero values")
display((numeric_check == 0).sum().rename("Zero_Count").to_frame())

print("\n4. Defective units exceeding quantity")
invalid_defects = df_raw.loc[
    numeric_check["Defective_Units"].notna()
    & numeric_check["Quantity"].notna()
    & (
        (numeric_check["Defective_Units"] < 0)
        | (numeric_check["Defective_Units"] > numeric_check["Quantity"])
    ),
    ["PO_ID", "Supplier", "Order_Status", "Quantity", "Defective_Units"]
]
display(invalid_defects)

print("Number of records:", len(invalid_defects))

print("\n5. Negotiated price higher than unit price")
price_anomalies = df_raw.loc[
    numeric_check["Negotiated_Price"].notna()
    & numeric_check["Unit_Price"].notna()
    & (
        numeric_check["Negotiated_Price"] > numeric_check["Unit_Price"]
    ),
    ["PO_ID", "Supplier", "Unit_Price", "Negotiated_Price"]
]
display(price_anomalies)

print("Number of records:", len(price_anomalies))

print("\n6. Numeric summary")
display(numeric_check.describe().T)


1. Numeric conversion failures
Quantity: 0
Unit_Price: 0
Negotiated_Price: 0
Defective_Units: 0

2. Negative values


,Negative_Count
Quantity,0
Unit_Price,0
Negotiated_Price,0
Defective_Units,0



3. Zero values


,Zero_Count
Quantity,0
Unit_Price,0
Negotiated_Price,0
Defective_Units,4



4. Defective units exceeding quantity


,PO_ID,Supplier,Order_Status,Quantity,Defective_Units


Number of records: 0

5. Negotiated price higher than unit price


,PO_ID,Supplier,Unit_Price,Negotiated_Price


Number of records: 0

6. Numeric summary


,count,mean,std,min,25%,50%,75%,max
Quantity,777.0,1094.660232,647.844551,51.00,615.00,1075.00,1548.00,5000.00
Unit_Price,777.0,58.283822,28.101315,10.84,33.29,58.95,83.13,109.17
Negotiated_Price,777.0,53.660721,26.094930,9.27,30.46,53.80,76.55,107.39
Defective_Units,641.0,74.803432,69.187870,0.00,26.00,49.00,100.00,321.00


### Step 12 — Interpret the numeric summary
Field	What we know	What needs checking
Quantity	777 values; range 51–5,000	Whether quantities are whole numbers and whether any values are negative or zero
Unit_Price	777 values; range 10.84–109.17	Whether all prices are positive and expressed in the same currency
Negotiated_Price	777 values; range 9.27–107.39	Whether negotiated prices are consistently less than or equal to unit prices
Defective_Units	641 non-missing values; range 0–321	Whether counts are whole numbers and no greater than quantity

The mean defect count of 74.80 applies only to the 641 records where the value is present. It is not the average defect count for all 777 orders, because we haven't established why the other 136 values are missing.

Also, a defect count of zero is valid and must remain distinct from a missing value.

### Step 13 — Check numeric anomalies

In [13]:

# Final numeric anomaly summary

num = df_raw[
    ["Quantity", "Unit_Price", "Negotiated_Price", "Defective_Units"]
].apply(pd.to_numeric, errors="coerce")

checks = {
    "Negative quantity": num["Quantity"] < 0,
    "Zero quantity": num["Quantity"] == 0,
    "Negative unit price": num["Unit_Price"] < 0,
    "Zero unit price": num["Unit_Price"] == 0,
    "Negative negotiated price": num["Negotiated_Price"] < 0,
    "Zero negotiated price": num["Negotiated_Price"] == 0,
    "Negotiated price above unit price": (
        num["Negotiated_Price"] > num["Unit_Price"]
    ),
    "Negative defect count": num["Defective_Units"] < 0,
    "Defect count above quantity": (
        num["Defective_Units"] > num["Quantity"]
    ),
    "Non-integer quantity": (
        num["Quantity"].notna()
        & (num["Quantity"] % 1 != 0)
    ),
    "Non-integer defect count": (
        num["Defective_Units"].notna()
        & (num["Defective_Units"] % 1 != 0)
    )
}

summary = pd.DataFrame({
    "Issue": list(checks.keys()),
    "Records_Flagged": [int(mask.sum()) for mask in checks.values()]
})

display(summary)

# Show the rows for any nonzero issue
for issue, mask in checks.items():
    if mask.sum() > 0:
        print(f"\n{issue}:")
        display(
            df_raw.loc[
                mask.fillna(False),
                [
                    "PO_ID", "Supplier", "Order_Status",
                    "Quantity", "Unit_Price",
                    "Negotiated_Price", "Defective_Units"
                ]
            ].head(10)
        )


,Issue,Records_Flagged
0,Negative quantity,0
1,Zero quantity,0
2,Negative unit price,0
3,Zero unit price,0
4,Negative negotiated price,0
5,Zero negotiated price,0
6,Negotiated price above unit price,0
7,Negative defect count,0
8,Defect count above quantity,0
9,Non-integer quantity,0


### Step 14 - Check the business logic of the records
Next, we'll investigate whether the order dates, delivery dates, and order statuses make sense together. This matters because our eventual dashboard should not report misleading supplier performance.

In [14]:

# Check whether order dates, delivery dates, and statuses are consistent

audit = df_raw.copy()

audit["Order_Date"] = pd.to_datetime(
    audit["Order_Date"], errors="coerce"
)
audit["Delivery_Date"] = pd.to_datetime(
    audit["Delivery_Date"], errors="coerce"
)

# Flag records where delivery occurred before the order date
audit["Delivery_Before_Order"] = (
    audit["Delivery_Date"] < audit["Order_Date"]
)

# Flag records marked as delivered but missing a delivery date
audit["Delivered_Missing_Date"] = (
    audit["Order_Status"].eq("Delivered")
    & audit["Delivery_Date"].isna()
)

# Flag cancelled orders that have a delivery date
audit["Cancelled_With_Date"] = (
    audit["Order_Status"].eq("Cancelled")
    & audit["Delivery_Date"].notna()
)

# Summarize the checks
business_checks = {
    "Delivery date before order date":
        audit["Delivery_Before_Order"],
    "Delivered orders missing delivery date":
        audit["Delivered_Missing_Date"],
    "Cancelled orders with delivery date":
        audit["Cancelled_With_Date"],
}

business_summary = pd.DataFrame({
    "Issue": list(business_checks.keys()),
    "Records_Flagged": [
        int(mask.fillna(False).sum())
        for mask in business_checks.values()
    ]
})

display(business_summary)

# Inspect any flagged records
for issue, mask in business_checks.items():
    if mask.fillna(False).any():
        print(f"\n{issue}:")
        display(
            audit.loc[
                mask.fillna(False),
                [
                    "PO_ID", "Supplier", "Order_Date",
                    "Delivery_Date", "Order_Status"
                ]
            ].head(10)
        )


,Issue,Records_Flagged
0,Delivery date before order date,1
1,Delivered orders missing delivery date,68
2,Cancelled orders with delivery date,55



Delivery date before order date:


,PO_ID,Supplier,Order_Date,Delivery_Date,Order_Status
100,PO-00101,Alpha_Inc,2022-02-27,2022-02-22,Delivered



Delivered orders missing delivery date:


,PO_ID,Supplier,Order_Date,Delivery_Date,Order_Status
13,PO-00014,Beta_Supplies,2022-02-02,NaT,Delivered
29,PO-00030,Epsilon_Group,2023-08-27,NaT,Delivered
39,PO-00040,Epsilon_Group,2022-12-11,NaT,Delivered
45,PO-00046,Alpha_Inc,2022-12-19,NaT,Delivered
51,PO-00052,Delta_Logistics,2022-05-08,NaT,Delivered
56,PO-00057,Delta_Logistics,2023-10-06,NaT,Delivered
78,PO-00079,Alpha_Inc,2022-08-03,NaT,Delivered
89,PO-00090,Alpha_Inc,2022-09-08,NaT,Delivered
98,PO-00099,Beta_Supplies,2023-12-03,NaT,Delivered
101,PO-00102,Alpha_Inc,2022-08-23,NaT,Delivered



Cancelled orders with delivery date:


,PO_ID,Supplier,Order_Date,Delivery_Date,Order_Status
0,PO-00001,Alpha_Inc,2023-10-17,2023-10-25,Cancelled
20,PO-00021,Epsilon_Group,2022-01-28,2022-02-12,Cancelled
53,PO-00054,Delta_Logistics,2022-03-22,2022-03-28,Cancelled
70,PO-00071,Alpha_Inc,2023-01-25,2023-02-08,Cancelled
82,PO-00083,Beta_Supplies,2023-12-01,2023-12-07,Cancelled
105,PO-00106,Delta_Logistics,2022-10-02,2022-10-22,Cancelled
120,PO-00121,Beta_Supplies,2023-07-29,2023-08-14,Cancelled
123,PO-00124,Alpha_Inc,2023-08-22,2023-08-25,Cancelled
125,PO-00126,Alpha_Inc,2023-08-21,2023-09-03,Cancelled
132,PO-00133,Delta_Logistics,2022-04-04,2022-04-22,Cancelled


### Step 15 : Get the complete business-check summary

In [15]:
# Show the total count for each business-logic check
for issue, mask in business_checks.items():
    print(f"{issue}: {int(mask.fillna(False).sum())} records")

# Count cancelled orders with and without delivery dates
cancelled = audit[audit["Order_Status"].eq("Cancelled")]

print("\nCancelled order breakdown:")
display(
    pd.DataFrame({
        "Category": [
            "Cancelled orders with delivery dates",
            "Cancelled orders without delivery dates"
        ],
        "Count": [
            int(cancelled["Delivery_Date"].notna().sum()),
            int(cancelled["Delivery_Date"].isna().sum())
        ]
    })
)


Delivery date before order date: 1 records
Delivered orders missing delivery date: 68 records
Cancelled orders with delivery date: 55 records

Cancelled order breakdown:


,Category,Count
0,Cancelled orders with delivery dates,55
1,Cancelled orders without delivery dates,8


### Step 16  — inspect the one date inconsistency

In [16]:
# Inspect the order whose delivery date is before its order date

date_issue = audit.loc[
    audit["Delivery_Before_Order"].fillna(False),
    [
        "PO_ID",
        "Supplier",
        "Order_Date",
        "Delivery_Date",
        "Order_Status",
        "Quantity",
        "Unit_Price",
        "Negotiated_Price",
        "Defective_Units",
        "Compliance"
    ]
]

display(date_issue)


,PO_ID,Supplier,Order_Date,Delivery_Date,Order_Status,Quantity,Unit_Price,Negotiated_Price,Defective_Units,Compliance
100,PO-00101,Alpha_Inc,2022-02-27,2022-02-22,Delivered,211,75.69,65.4,2.0,Yes


### Step 17 - Document the audit findings

In [17]:
# Record the data-quality issues identified during the audit

issue_register = pd.DataFrame([
    {
        "Issue": "Missing Delivery_Date",
        "Records_Affected": 87,
        "Decision": "Retain missing values; investigate before calculating delivery KPIs"
    },
    {
        "Issue": "Missing Defective_Units",
        "Records_Affected": 136,
        "Decision": "Retain missing values; do not assume zero defects"
    },
    {
        "Issue": "Delivered orders missing Delivery_Date",
        "Records_Affected": 68,
        "Decision": "Exclude from date-based delivery metrics until resolved"
    },
    {
        "Issue": "Cancelled orders with Delivery_Date",
        "Records_Affected": 55,
        "Decision": "Flag for investigation; do not change status automatically"
    },
    {
        "Issue": "Delivery_Date before Order_Date",
        "Records_Affected": 1,
        "Decision": "Flag PO-00101; verify source dates before correction"
    }
])

display(issue_register)


,Issue,Records_Affected,Decision
0,Missing Delivery_Date,87,Retain missing values; investigate before calc...
1,Missing Defective_Units,136,Retain missing values; do not assume zero defects
2,Delivered orders missing Delivery_Date,68,Exclude from date-based delivery metrics until...
3,Cancelled orders with Delivery_Date,55,Flag for investigation; do not change status a...
4,Delivery_Date before Order_Date,1,Flag PO-00101; verify source dates before corr...


## Phase 2 — Data Cleaning and Validation

## Step 1 - Validation of the working copy

In [18]:
# Create a working copy; preserve df_raw unchanged
df_clean = df_raw.copy()

# Standardize date columns in the working copy
df_clean["Order_Date"] = pd.to_datetime(
    df_clean["Order_Date"], errors="coerce"
)

df_clean["Delivery_Date"] = pd.to_datetime(
    df_clean["Delivery_Date"], errors="coerce"
)

# Remove accidental whitespace from text fields
text_columns = [
    "PO_ID",
    "Supplier",
    "Item_Category",
    "Order_Status",
    "Compliance"
]

for column in text_columns:
    df_clean[column] = df_clean[column].apply(
        lambda value: value.strip()
        if isinstance(value, str)
        else value
    )

# Add transparent data-quality flags
df_clean["Missing_Delivery_Date"] = (
    df_clean["Delivery_Date"].isna()
)

df_clean["Missing_Defective_Units"] = (
    df_clean["Defective_Units"].isna()
)

df_clean["Delivery_Before_Order"] = (
    df_clean["Delivery_Date"] < df_clean["Order_Date"]
)

df_clean["Cancelled_With_Delivery_Date"] = (
    df_clean["Order_Status"].eq("Cancelled")
    & df_clean["Delivery_Date"].notna()
)

# Confirm dimensions and inspect the result
print("Raw dataset shape:", df_raw.shape)
print("Working dataset shape:", df_clean.shape)

display(df_clean.head())


Raw dataset shape: (777, 11)
Working dataset shape: (777, 15)


,PO_ID,Supplier,Order_Date,Delivery_Date,Item_Category,Order_Status,Quantity,Unit_Price,Negotiated_Price,Defective_Units,Compliance,Missing_Delivery_Date,Missing_Defective_Units,Delivery_Before_Order,Cancelled_With_Delivery_Date
0,PO-00001,Alpha_Inc,2023-10-17,2023-10-25,Office Supplies,Cancelled,1176,20.13,17.81,NaN,Yes,False,True,False,True
1,PO-00002,Delta_Logistics,2022-04-25,2022-05-05,Office Supplies,Delivered,1509,39.32,37.34,235.0,Yes,False,False,False,False
2,PO-00003,Gamma_Co,2022-01-26,2022-02-15,MRO,Delivered,910,95.51,92.26,41.0,Yes,False,False,False,False
3,PO-00004,Beta_Supplies,2022-10-09,2022-10-28,Packaging,Delivered,1344,99.85,95.52,112.0,Yes,False,False,False,False
4,PO-00005,Delta_Logistics,2022-09-08,2022-09-20,Raw Materials,Delivered,1180,64.07,60.53,171.0,No,False,False,False,False


## Step 2: Validate the new quality flags

In [19]:
# Check whether all four quality flags contain True/False values

flag_columns = [
    "Missing_Delivery_Date",
    "Missing_Defective_Units",
    "Delivery_Before_Order",
    "Cancelled_With_Delivery_Date"
]

print("Missing values in quality flags:")
display(df_clean[flag_columns].isna().sum().to_frame("Missing_Count"))

print("\nFlag counts:")
for column in flag_columns:
    print(f"\n{column}")
    display(df_clean[column].value_counts(dropna=False).to_frame("Records"))

# Confirm the original dataset was not modified
print("\nOriginal dataset shape:", df_raw.shape)
print("Working dataset shape:", df_clean.shape)


Missing values in quality flags:


,Missing_Count
Missing_Delivery_Date,0
Missing_Defective_Units,0
Delivery_Before_Order,0
Cancelled_With_Delivery_Date,0



Flag counts:

Missing_Delivery_Date


,Records
Missing_Delivery_Date,
False,690
True,87



Missing_Defective_Units


,Records
Missing_Defective_Units,
False,641
True,136



Delivery_Before_Order


,Records
Delivery_Before_Order,
False,776
True,1



Cancelled_With_Delivery_Date


,Records
Cancelled_With_Delivery_Date,
False,722
True,55



Original dataset shape: (777, 11)
Working dataset shape: (777, 15)


## Step 3: Create the first derived business metrics
Before we clean or exclude any records, let's calculate three useful procurement metrics in the working copy:
1. Delivery duration — days between the order date and delivery date.
2. Potential negotiated savings — quantity multiplied by the difference between unit price and negotiated price.
3. Defect rate — defective units divided by quantity, expressed as a percentage.
We'll preserve missing values and flag invalid delivery durations rather than silently treating them as valid.

In [20]:
# Create derived procurement metrics

# 1. Delivery duration in days
df_clean["Delivery_Duration_Days"] = (
    df_clean["Delivery_Date"] - df_clean["Order_Date"]
).dt.days

# 2. Flag delivery durations that cannot be used as valid durations
df_clean["Invalid_Delivery_Duration"] = (
    df_clean["Delivery_Duration_Days"] < 0
)

# 3. Potential negotiated savings per unit
df_clean["Savings_Per_Unit"] = (
    df_clean["Unit_Price"] - df_clean["Negotiated_Price"]
)

# 4. Potential total negotiated savings
df_clean["Potential_Total_Savings"] = (
    df_clean["Quantity"] * df_clean["Savings_Per_Unit"]
)

# 5. Defect rate as a percentage
df_clean["Defect_Rate_Pct"] = (
    df_clean["Defective_Units"] / df_clean["Quantity"] * 100
)

# 6. Summarize metric availability
metric_columns = [
    "Delivery_Duration_Days",
    "Savings_Per_Unit",
    "Potential_Total_Savings",
    "Defect_Rate_Pct"
]

print("Metric availability:")
display(
    pd.DataFrame({
        "Available_Values": df_clean[metric_columns].notna().sum(),
        "Missing_Values": df_clean[metric_columns].isna().sum()
    })
)

print("\nDelivery duration validation:")
print(
    "Negative durations:",
    int(df_clean["Invalid_Delivery_Duration"].sum())
)

print("\nPreview of derived metrics:")
display(
    df_clean[
        [
            "PO_ID",
            "Order_Status",
            "Delivery_Duration_Days",
            "Savings_Per_Unit",
            "Potential_Total_Savings",
            "Defect_Rate_Pct"
        ]
    ].head(10)
)


Metric availability:


,Available_Values,Missing_Values
Delivery_Duration_Days,690,87
Savings_Per_Unit,777,0
Potential_Total_Savings,777,0
Defect_Rate_Pct,641,136



Delivery duration validation:
Negative durations: 1

Preview of derived metrics:


,PO_ID,Order_Status,Delivery_Duration_Days,Savings_Per_Unit,Potential_Total_Savings,Defect_Rate_Pct
0,PO-00001,Cancelled,8.0,2.32,2728.32,NaN
1,PO-00002,Delivered,10.0,1.98,2987.82,15.573227
2,PO-00003,Delivered,20.0,3.25,2957.50,4.505495
3,PO-00004,Delivered,19.0,4.33,5819.52,8.333333
4,PO-00005,Delivered,12.0,3.54,4177.20,14.491525
5,PO-00006,Delivered,12.0,5.64,6457.80,3.406114
6,PO-00007,Delivered,11.0,3.55,6297.70,5.411499
7,PO-00008,Delivered,14.0,4.15,4540.10,2.010969
8,PO-00009,Partially Delivered,4.0,4.04,6819.52,5.272512
9,PO-00010,Pending,12.0,6.67,1140.57,4.678363


### Interpret the derived metrics
## Delivery duration
690 / 777 available
87 orders have no delivery date. One additional record has a negative duration and should not be used as a valid delivery duration.
## Potential negotiated savings
777 / 777 available
The arithmetic is available for every order, but potential savings should not be treated as realized savings, particularly for cancelled orders.
## Defect rate
641 / 777 available
136 orders lack defect counts. Keep their defect rates missing rather than assuming zero defects.

One detail: the 690 available delivery durations include the one negative duration. For valid delivery-time analysis, exclude that record from the calculation until its dates are verified.

### Step 4: Define KPI eligibility

In [21]:
# Create metric eligibility flags
# Valid delivery duration:
# date exists, duration is non-negative, and order is not cancelled.
df_clean["Eligible_Delivery_KPI"] = (
    df_clean["Delivery_Duration_Days"].notna()
    & ~df_clean["Invalid_Delivery_Duration"]
    & df_clean["Order_Status"].ne("Cancelled")
)

# Defect-rate eligibility:
# defect count is available and the order is not cancelled.
df_clean["Eligible_Defect_KPI"] = (
    df_clean["Defective_Units"].notna()
    & df_clean["Order_Status"].ne("Cancelled")
)

# Potential savings is calculated for all records.
# This flag identifies non-cancelled records for a separate
# operational analysis; it does not prove savings were realized.
df_clean["Eligible_Potential_Savings_Analysis"] = (
    df_clean["Order_Status"].ne("Cancelled")
)

eligibility_summary = pd.DataFrame({
    "Metric": [
        "Delivery duration",
        "Defect rate",
        "Potential savings analysis"
    ],
    "Eligible_Records": [
        int(df_clean["Eligible_Delivery_KPI"].sum()),
        int(df_clean["Eligible_Defect_KPI"].sum()),
        int(df_clean["Eligible_Potential_Savings_Analysis"].sum())
    ],
    "Excluded_Records": [
        int((~df_clean["Eligible_Delivery_KPI"]).sum()),
        int((~df_clean["Eligible_Defect_KPI"]).sum()),
        int((~df_clean["Eligible_Potential_Savings_Analysis"]).sum())
    ]
})

display(eligibility_summary)


,Metric,Eligible_Records,Excluded_Records
0,Delivery duration,634,143
1,Defect rate,587,190
2,Potential savings analysis,714,63


### Interpret KPI eligibility
Metric	Eligible	Excluded	Main reason for exclusion
Delivery duration	634	143	Missing dates, negative duration, or cancelled status
Defect rate	587	190	Missing defect count or cancelled status
Potential savings analysis	714	63	Cancelled status
These exclusions are based on our provisional rules, not necessarily the organisation's official reporting policy.
One important distinction: excluded from a KPI does not mean deleted from the dataset. All 777 purchase orders should remain available for appropriate analysis.

### Step 5: Validate potential savings before export
Let's check that the calculated savings are non-negative, that the negotiated price never exceeds the listed price, and that our calculations match the source fields.

In [23]:
# Recalculate the expected values independently
expected_savings_per_unit = (
    df_clean["Unit_Price"] - df_clean["Negotiated_Price"]
)

expected_total_savings = (
    df_clean["Quantity"] * expected_savings_per_unit
)

# Compare calculated and expected values
savings_per_unit_mismatch = ~np.isclose(
    df_clean["Savings_Per_Unit"],
    expected_savings_per_unit
)

total_savings_mismatch = ~np.isclose(
    df_clean["Potential_Total_Savings"],
    expected_total_savings
)

# Count the issues correctly
validation = pd.DataFrame({
    "Check": [
        "Savings per unit matches recalculation",
        "Total savings matches recalculation",
        "Negative savings per unit",
        "Negotiated price above unit price",
        "Missing potential total savings"
    ],
    "Records_Flagged": [
        int(savings_per_unit_mismatch.sum()),
        int(total_savings_mismatch.sum()),
        int((df_clean["Savings_Per_Unit"] < 0).sum()),
        int(
            (
                df_clean["Negotiated_Price"]
                > df_clean["Unit_Price"]
            ).sum()
        ),
        int(df_clean["Potential_Total_Savings"].isna().sum())
    ]
})

display(validation)

print(
    "\nTotal potential savings across all records:",
    round(df_clean["Potential_Total_Savings"].sum(), 2)
)

print(
    "Potential savings for non-cancelled orders:",
    round(
        df_clean.loc[
            df_clean["Eligible_Potential_Savings_Analysis"],
            "Potential_Total_Savings"
        ].sum(),
        2
    )
)


,Check,Records_Flagged
0,Savings per unit matches recalculation,0
1,Total savings matches recalculation,0
2,Negative savings per unit,0
3,Negotiated price above unit price,0
4,Missing potential total savings,0



Total potential savings across all records: 3931126.47
Potential savings for non-cancelled orders: 3638920.08


### Step 6: Save the cleaned dataset and audit reports

In [24]:
from pathlib import Path

OUTPUT_DIR = PROJECT_DIR / "outputs"
DOCS_DIR = PROJECT_DIR / "docs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

# 1. Export cleaned working dataset
clean_path = OUTPUT_DIR / "procurement_data_cleaned.csv"
df_clean.to_csv(clean_path, index=False)

# 2. Export original missing-value audit
missing_path = OUTPUT_DIR / "raw_missingness_audit.csv"
missing_audit.to_csv(missing_path)

# 3. Export issue register
issues_path = DOCS_DIR / "data_quality_issue_register.csv"
issue_register.to_csv(issues_path, index=False)

# 4. Export KPI eligibility summary
eligibility_path = OUTPUT_DIR / "kpi_eligibility_summary.csv"
eligibility_summary.to_csv(eligibility_path, index=False)

# 5. Export savings validation report
validation_path = OUTPUT_DIR / "savings_validation.csv"
validation.to_csv(validation_path, index=False)

print("Files saved successfully:")
for path in [
    clean_path,
    missing_path,
    issues_path,
    eligibility_path,
    validation_path
]:
    print(f"{path} | Exists: {path.exists()}")

print("\nCleaned dataset shape:", df_clean.shape)


Files saved successfully:
C:\Users\User\Desktop\Procurement KPI Analysis Dataset\outputs\procurement_data_cleaned.csv | Exists: True
C:\Users\User\Desktop\Procurement KPI Analysis Dataset\outputs\raw_missingness_audit.csv | Exists: True
C:\Users\User\Desktop\Procurement KPI Analysis Dataset\docs\data_quality_issue_register.csv | Exists: True
C:\Users\User\Desktop\Procurement KPI Analysis Dataset\outputs\kpi_eligibility_summary.csv | Exists: True
C:\Users\User\Desktop\Procurement KPI Analysis Dataset\outputs\savings_validation.csv | Exists: True

Cleaned dataset shape: (777, 23)


### Step 7: Verify the saved dataset

In [25]:
df_export_check = pd.read_csv(clean_path)

print("Exported dataset shape:", df_export_check.shape)

print("\nDuplicate purchase order IDs:")
print(df_export_check["PO_ID"].duplicated().sum())

print("\nMissing delivery dates:")
print(df_export_check["Missing_Delivery_Date"].sum())

print("\nMissing defect counts:")
print(df_export_check["Missing_Defective_Units"].sum())

print("\nNegative delivery durations:")
print(df_export_check["Invalid_Delivery_Duration"].sum())

print("\nRequired columns present:")
required_columns = [
    "PO_ID",
    "Supplier",
    "Order_Status",
    "Potential_Total_Savings",
    "Defect_Rate_Pct",
    "Eligible_Delivery_KPI",
    "Eligible_Defect_KPI"
]

for column in required_columns:
    print(f"{column}: {column in df_export_check.columns}")


Exported dataset shape: (777, 23)

Duplicate purchase order IDs:
0

Missing delivery dates:
87

Missing defect counts:
136

Negative delivery durations:
1

Required columns present:
PO_ID: True
Supplier: True
Order_Status: True
Potential_Total_Savings: True
Defect_Rate_Pct: True
Eligible_Delivery_KPI: True
Eligible_Defect_KPI: True
